Fused Matrix Multiplication with Residual, Bias, and ReLU
Advanced · Difficulty 9/10
Your task is to build a GPU kernel using Triton that performs matrix multiplication and three additional operations in a single kernel launch.

This problem tests whether you understand 2D GPU tiling, pointer arithmetic, broadcasting, strides, reduction loops, masking, and memory access.

You are expected to implement the kernel from scratch using Triton, following the pointer-increment approach from the official Triton matrix multiplication tutorial.
1. Problem statement
You are given four inputs:
- Matrix A of shape (M, K).
- Matrix B of shape (K, N).
- Residual matrix R of shape (M, N).
- Bias vector bias of shape (N,).
You are also given two scalar values, alpha and beta.

Your task is to compute an output matrix C of shape (M, N) using:
\[
\boxed{C=\operatorname{ReLU}(\alpha(A B)+\beta R+bias)}
\]

The bias vector is broadcast across every row of the output.

The ReLU activation is defined as:
\[
\operatorname{ReLU}(x)=\max(0,x)
\]

For any output element at row \(m\) and column \(n\), calculate:
\[
\boxed{
C[m,n]=\max\left(0,\alpha\sum_{k=0}^{K-1}A[m,k]B[k,n]+\beta R[m,n]+bias[n]\right)
}
\]

You must perform all the operations in one fused Triton kernel, without materializing intermediate matrices in global memory.

In [2]:

import torch
import triton
import triton.language as tl


In [ ]:

@triton.jit
def fused_gemm_kernel(
    A, B, R, Bias, C,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    stride_am: tl.constexpr,
    stride_ak: tl.constexpr,

    stride_bk: tl.constexpr,
    stride_bn: tl.constexpr,

    stride_rm: tl.constexpr,
    stride_rn: tl.constexpr,

    stride_bias: tl.constexpr,

    stride_cm: tl.constexpr,
    stride_cn: tl.constexpr,

    ALPHA: tl.constexpr,
    BETA: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    # -------------------------------------------------
    # STEP 1: Program IDs
    # -------------------------------------------------

    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    m = pid_m * BM + tl.arange(0, BM)
    n = pid_n * BN + tl.arange(0, BN)
    k = tl.arange(0, BK)

    # -------------------------------------------------
    # TODO 1: Construct A pointer tensor
    #
    # Shape: (BM, BK)
    # Use m, k, stride_am, stride_ak.
    # Initialize OUTSIDE the K loop.
    # -------------------------------------------------

    a_ptrs =  A+(m[:,None]*stride_am+k[None,:]*stride_ak)

    # -------------------------------------------------
    # TODO 2: Construct B pointer tensor
    #
    # Shape: (BK, BN)
    # B may be non-contiguous.
    # Use k, n, stride_bk, stride_bn.
    # -------------------------------------------------

    b_ptrs = B +(n[None,:]*stride_bn+k[:,None]*stride_bk)

    # -------------------------------------------------
    # TODO 3: Initialize FP32 accumulator
    #
    # Shape: (BM, BN)
    # -------------------------------------------------

    acc = tl.zeros((BM,BN),dtype=tl.float32)

    # -------------------------------------------------
    # STEP 2: Reduction loop
    # -------------------------------------------------

    for t in range(tl.cdiv(K, BK)):

        # ---------------------------------------------
        # TODO 4: Load A tile
        #
        # Apply M and K boundary masks.
        # The pointers have already been initialized.
        # Do not reconstruct them.
        # ---------------------------------------------

        a = tl.load(a_ptrs,mask=((m[:,None]< M) & (t*BK+k[None,:] <K )),other=0.0)

        # ---------------------------------------------
        # TODO 5: Load B tile
        #
        # Apply K and N boundary masks.
        # ---------------------------------------------

        b = tl.load(b_ptrs,mask=((n[None,:]<N)&(t*BK+k[:,None]<K)),other=0.0)

        # ---------------------------------------------
        # TODO 6: Matrix multiplicationc
        #
        # Accumulate, do not overwrite previous sums.
        # ---------------------------------------------

        acc = tl.dot(a,b,acc=acc)

        # ---------------------------------------------
        # TODO 7: Advance both pointers along K
        #
        # Do NOT create a kk vector.
        # Think about the K stride of each matrix.
        # ---------------------------------------------

        a_ptrs += BK*stride_ak
        b_ptrs += BK*stride_bk

    # -------------------------------------------------
    # STEP 3: Fused epilogue
    # -------------------------------------------------

    # ---------------------------------------------
    # TODO 8: Load residual tile R
    #
    # R shape: (M, N)
    # Tile shape: (BM, BN)
    # R can be non-contiguous.
    # ---------------------------------------------
   
    
    r_ptrs = R + (m[:,None]*stride_rm+n[None,:]*stride_rn)

    residual = tl.load(r_ptrs,mask=((m[:,None]< M) & (n[None,:] < N)),other=0.0)

    # ---------------------------------------------
    # TODO 9: Load bias and fuse operations
    #
    # Bias shape: (N,)
    # Its stride may be greater than 1.
    #
    # Compute, in FP32:
    #
    # result = ALPHA * acc
    #        + BETA * residual
    #        + bias
    #
    # Then apply ReLU.
    #
    # Think carefully about bias broadcasting.
    # ---------------------------------------------

    bias_ptrs = Bias + n*stride_bias

    bias = tl.load(bias_ptrs,mask = n<N,other=0.0)

    result = tl.maximum(ALPHA * acc + BETA *residual + bias,0.0)

    # -------------------------------------------------
    # TODO 10: Store output
    #
    # C shape: (M, N)
    # Output pointer shape: (BM, BN)
    # C may be non-contiguous.
    # Mask both M and N boundaries.
    # -------------------------------------------------

    c_ptrs = C + (m[:,None]*stride_cm+n[None,:]*stride_cn)

    c_mask = ((m[:,None]<M) & (n[None,:]<N))

    tl.store(c_ptrs, result, mask=c_mask)


# =====================================================
# PART 2: PYTHON LAUNCHER — ALREADY PROVIDED
# =====================================================


In [4]:

def fused_gemm(A, B, R, bias, C,
               alpha=0.75, beta=0.20):

    M, K = A.shape
    Kb, N = B.shape

    assert K == Kb
    assert R.shape == (M, N)
    assert bias.shape == (N,)
    assert C.shape == (M, N)

    BM = 32
    BN = 32
    BK = 32

    grid = (
        triton.cdiv(M, BM),
        triton.cdiv(N, BN),
    )

    fused_gemm_kernel[grid](
        A, B, R, bias, C,

        M, N, K,

        *A.stride(),
        *B.stride(),
        *R.stride(),

        bias.stride(0),

        *C.stride(),

        ALPHA=alpha,
        BETA=beta,

        BM=BM,
        BN=BN,
        BK=BK,

        num_warps=4,
    )

    return C

In [5]:

# =====================================================
# PART 3: TEST CASE GENERATOR
# =====================================================

def make_inputs(M, N, K, layout):

    device = "cuda"
    dtype = torch.float16

    if layout == "contiguous":

        A = torch.randn(
            (M, K), device=device, dtype=dtype
        )

        B = torch.randn(
            (K, N), device=device, dtype=dtype
        )

        R = torch.randn(
            (M, N), device=device, dtype=dtype
        )

        bias = torch.randn(
            (N,), device=device, dtype=dtype
        )

    elif layout == "transposed":

        # Both A and B are non-contiguous views.
        A = torch.randn(
            (K, M), device=device, dtype=dtype
        ).T

        B = torch.randn(
            (N, K), device=device, dtype=dtype
        ).T

        R = torch.randn(
            (N, M), device=device, dtype=dtype
        ).T

        bias = torch.randn(
            (N * 2,), device=device, dtype=dtype
        )[::2]

    elif layout == "sliced":

        # Non-unit strides along multiple axes.
        A = torch.randn(
            (M, K * 2), device=device, dtype=dtype
        )[:, ::2]

        B = torch.randn(
            (K * 2, N * 2),
            device=device,
            dtype=dtype
        )[::2, ::2]

        R = torch.randn(
            (M, N * 2), device=device, dtype=dtype
        )[:, ::2]

        bias = torch.randn(
            (N * 2,), device=device, dtype=dtype
        )[::2]

    else:
        raise ValueError("Unknown layout")

    # Output is intentionally non-contiguous.
    C_storage = torch.empty(
        (M, N * 2), device=device, dtype=dtype
    )

    C = C_storage[:, ::2]

    return A, B, R, bias, C

In [6]:



# =====================================================
# PART 4: CORRECTNESS TESTS
# =====================================================

def run_tests():

    torch.manual_seed(42)

    # Use full FP32 precision for the reference.
    torch.backends.cuda.matmul.allow_tf32 = False

    cases = [
        # M, N, K, layout
        (32, 32, 32, "contiguous"),
        (70, 90, 75, "contiguous"),
        (70, 90, 75, "transposed"),
        (33, 65, 17, "transposed"),
        (63, 97, 33, "sliced"),
        (128, 64, 96, "sliced"),
        (1, 1, 1, "contiguous"),
    ]

    alpha = 0.75
    beta = 0.20

    for i, (M, N, K, layout) in enumerate(cases):

        A, B, R, bias, C = make_inputs(
            M, N, K, layout
        )

        actual = fused_gemm(
            A, B, R, bias, C,
            alpha=alpha,
            beta=beta,
        )

        # FP32 reference, converted to FP16 at end.
        expected_fp32 = (
            alpha * (A.float() @ B.float())
            + beta * R.float()
            + bias.float()[None, :]
        )

        expected = torch.relu(
            expected_fp32
        ).to(torch.float16)

        torch.testing.assert_close(
            actual,
            expected,
            atol=2e-2,
            rtol=2e-2,
        )

        print(
            f"PASS test {i+1}: "
            f"M={M}, N={N}, K={K}, "
            f"layout={layout}"
        )

    print("\nALL TESTS PASSED!")


if __name__ == "__main__":
    assert torch.cuda.is_available(), \
        "This challenge requires a CUDA GPU."

    run_tests()


PASS test 1: M=32, N=32, K=32, layout=contiguous
PASS test 2: M=70, N=90, K=75, layout=contiguous
PASS test 3: M=70, N=90, K=75, layout=transposed
PASS test 4: M=33, N=65, K=17, layout=transposed
PASS test 5: M=63, N=97, K=33, layout=sliced
PASS test 6: M=128, N=64, K=96, layout=sliced
PASS test 7: M=1, N=1, K=1, layout=contiguous

ALL TESTS PASSED!
